In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("D:/ML PIPELINE/kidney_disease.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.tail())
print(df.info())
print(df.describe())

Dataset shape: (400, 26)
   id   age    bp     sg   al   su     rbc        pc         pcc          ba  \
0   0  48.0  80.0  1.020  1.0  0.0     NaN    normal  notpresent  notpresent   
1   1   7.0  50.0  1.020  4.0  0.0     NaN    normal  notpresent  notpresent   
2   2  62.0  80.0  1.010  2.0  3.0  normal    normal  notpresent  notpresent   
3   3  48.0  70.0  1.005  4.0  0.0  normal  abnormal     present  notpresent   
4   4  51.0  80.0  1.010  2.0  0.0  normal    normal  notpresent  notpresent   

   ...  pcv    wc   rc  htn   dm  cad appet   pe  ane classification  
0  ...   44  7800  5.2  yes  yes   no  good   no   no            ckd  
1  ...   38  6000  NaN   no   no   no  good   no   no            ckd  
2  ...   31  7500  NaN   no  yes   no  poor   no  yes            ckd  
3  ...   32  6700  3.9  yes   no   no  poor  yes  yes            ckd  
4  ...   35  7300  4.6   no   no   no  good   no   no            ckd  

[5 rows x 26 columns]
      id   age    bp     sg   al   su     rbc

In [4]:
# Clean column names
df.columns = df.columns.str.strip()

# Clean spaces and tabs in text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Replace question marks and empty strings with missing values
df = df.replace(r"^\s*\?$", np.nan, regex=True)
df = df.replace(r"^\s*$", np.nan, regex=True)

# Convert numeric columns stored as text
numeric_cols_to_convert = [
    "age", "bp", "sg", "al", "su",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wc", "rc"
]

for col in numeric_cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove identifier
df = df.drop(columns=["id"], errors="ignore")

# Clean target labels
df["classification"] = (
    df["classification"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Remove rows with missing target
df = df.dropna(subset=["classification"]).copy()

print("Shape after cleaning:", df.shape)
print("\nTarget distribution:")
print(df["classification"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Shape after cleaning: (400, 25)

Target distribution:
classification
ckd       250
notckd    150
Name: count, dtype: Int64

Missing values:
age                 9
bp                 12
sg                 47
al                 46
su                 49
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                44
bu                 19
sc                 17
sod                87
pot                88
hemo               52
pcv                71
wc                106
rc                131
htn                 2
dm                  2
cad                 2
appet               1
pe                  1
ane                 1
classification      0
dtype: int64


In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Features (input columns)
X = df.drop(columns=["classification"])

# Target (output column)
y = df["classification"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (400, 24)
y shape: (400,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (320, 24)
Testing data: (80, 24)


In [8]:
num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

cat_cols = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']
Categorical columns: ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

print("All ML algorithms imported successfully!")

All ML algorithms imported successfully!


In [11]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200, random_state=42
    ),

    "SVM": SVC(kernel="rbf"),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Naive Bayes": GaussianNB(),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    )
}

print("Total models:", len(models))

for name in models:
    print(name)

Total models: 9
Logistic Regression
Decision Tree
Random Forest
Extra Trees
SVM
KNN
Naive Bayes
Gradient Boosting
AdaBoost


In [12]:
from sklearn.pipeline import Pipeline

pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

print("All ML pipelines created successfully!")

All ML pipelines created successfully!


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
import pandas as pd

# 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score,
        pos_label="ckd",
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        pos_label="ckd",
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        pos_label="ckd",
        zero_division=0
    )
}

results = []

for name, pipeline in pipelines.items():
    print(f"Training {name}...")

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1-Score": scores["test_f1"].mean()
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="CV F1-Score",
    ascending=False
).reset_index(drop=True)

print("\nModel Comparison Results:")
print(results_df.round(4).to_string(index=False))

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training Extra Trees...
Training SVM...
Training KNN...
Training Naive Bayes...
Training Gradient Boosting...
Training AdaBoost...

Model Comparison Results:
              Model  CV Accuracy  CV Precision  CV Recall  CV F1-Score
        Extra Trees       0.9969        1.0000      0.995       0.9975
Logistic Regression       0.9938        0.9951      0.995       0.9950
      Random Forest       0.9906        0.9905      0.995       0.9926
                SVM       0.9906        1.0000      0.985       0.9924
           AdaBoost       0.9812        0.9947      0.975       0.9847
  Gradient Boosting       0.9781        0.9902      0.975       0.9823
      Decision Tree       0.9719        0.9805      0.975       0.9774
                KNN       0.9719        1.0000      0.955       0.9767
        Naive Bayes       0.9625        1.0000      0.940       0.9682


In [14]:
results_df.to_csv(
    "kidney_model_comparison.csv",
    index=False
)

print("Model comparison saved successfully!")

Model comparison saved successfully!


In [21]:
best_model_name = results_df.iloc[0]["Model"]

best_pipeline = pipelines[best_model_name]

print("Best Model:", best_model_name)

Best Model: Extra Trees


In [22]:
# Train the best model using the training data
best_pipeline.fit(X_train, y_train)

print("Best model trained successfully!")

# Save the complete trained pipeline
joblib.dump(
    best_pipeline,
    "kidney_disease_best_pipeline.pkl"
)

print("Model saved successfully!")

Best model trained successfully!
Model saved successfully!


In [23]:
sample = X_test.iloc[[0]]

prediction = best_pipeline.predict(sample)

print("Predicted class:", prediction[0])
print("Actual class:", y_test.iloc[0])

Predicted class: ckd
Actual class: ckd
